In [1]:
# Classificação Automática de Atividades Culturais e Educacionais
# 08 - Conferência humana das categorias
#
# Integrantes: Beatriz Aparecida de Mello Barbosa (10354067), Bruna Gonçalves Corte David (10425696),
#              Henrique Brainer Costa (10420717), João Pedro Queiroz de Andrade (10425822),
#              Júlia Andrade (10428513)
#
# Conteúdo: sorteia 100 atividades do teste, divide entre os cinco integrantes de um jeito em que
#           metade delas é vista por duas pessoas e gera a planilha e as instruções de cada um para
#           rotular as atividades do zero. Depois lê as respostas que voltaram, monta o gabarito
#           humano, mede a concordância entre as pessoas e compara a régua com o gabarito.
#
# Alterações:
#   11/10/2026 - Bruna - notebook novo, com o sorteio da amostra e a roda
#   13/10/2026 - Bruna - planilhas e instruções para rotular as atividades do zero
#   27/10/2026 - Beatriz, Bruna, Henrique, João e Júlia - rotulagem do zero das 30 atividades de
#                        cada um, que formam o gabarito humano
#   27/10/2026 - Bruna - leitura das respostas, gabarito e comparação com o modelo de linguagem

As categorias do notebook 07 saíram de uma conta, e quem escolheu a regra foi um juiz que usa os mesmos
embeddings, então, para saber se elas fazem sentido para quem lê, precisávamos ler as atividades nós
mesmos. Como ler as 693 não cabia no nosso tempo, rotulamos uma amostra do teste: cada um de nós leu as
atividades e escolheu as categorias do zero, em ordem, sem ver o que a régua deu, para o gabarito que
sai daqui não depender da régua.

Para medir se o critério está claro, metade das atividades foi rotulada por dois de nós, sem um ver a
resposta do outro. Organizamos isso numa roda, com os cinco integrantes em ordem alfabética, cada um com
30 atividades, sendo 10 só dele, 10 em comum com o integrante seguinte e 10 em comum com o anterior, e o
último fecha a roda com o primeiro. No total são 100 atividades, 50 delas rotuladas em dobro.

In [2]:
# ====================================================================================================
#                                CARREGAMENTO DAS ATIVIDADES DE TESTE
# ====================================================================================================
import pandas as pd   # leitura e manipulação das tabelas
import numpy as np    # sorteio e contas com vetores

activities_multilabel = pd.read_csv("../data/03_processed/07_activities_multilabel.csv", sep=";", encoding="utf-8-sig")  # tabela gerada pelo notebook 07
test_activities = activities_multilabel[activities_multilabel["split"] == "test"]                                        # a amostra sai só do teste

print("Atividades de teste:", len(test_activities))

Atividades de teste: 139


In [3]:
# ====================================================================================================
#                            SORTEIO DAS 100 ATIVIDADES E MONTAGEM DA RODA
# ====================================================================================================
random_generator = np.random.default_rng(42)   # semente fixa: o sorteio é sempre o mesmo

sampled_ids = random_generator.choice(test_activities["activity_id"].values, 100, replace=False)   # 100 das 139, já em ordem sorteada
blocks_of_ten = sampled_ids.reshape(10, 10)                                                        # 10 blocos de 10 atividades

reviewers = ["Beatriz", "Bruna", "Henrique", "João", "Júlia"]   # a roda, em ordem alfabética
number_of_reviewers = len(reviewers)                            # cinco pessoas

ids_of_each_reviewer = {}                                                                                   # pessoa -> as 30 atividades dela
for reviewer_position, reviewer in enumerate(reviewers):
    previous_position = (reviewer_position - 1) % number_of_reviewers                                       # quem vem antes na roda (a primeira volta para a última)
    own_block = blocks_of_ten[reviewer_position]                                                            # 10 atividades só dela
    block_shared_with_next = blocks_of_ten[number_of_reviewers + reviewer_position]                         # 10 em comum com a pessoa seguinte
    block_shared_with_previous = blocks_of_ten[number_of_reviewers + previous_position]                     # 10 em comum com a anterior
    ids_of_the_reviewer = np.concatenate([own_block, block_shared_with_next, block_shared_with_previous])   # as 30 juntas
    ids_of_each_reviewer[reviewer] = random_generator.permutation(ids_of_the_reviewer)                      # embaralha, para não dar para saber quais são em dobro

review_assignment = []                                                                                                                                                  # uma linha por atividade sorteada, com quem confere
for activity_id in sampled_ids:
    reviewers_of_the_activity = [reviewer for reviewer in reviewers if activity_id in ids_of_each_reviewer[reviewer]]                                                   # uma ou duas pessoas
    review_assignment.append({"activity_id": activity_id, "reviewers": " | ".join(reviewers_of_the_activity), "number_of_reviewers": len(reviewers_of_the_activity)})   # guarda quem ficou com a atividade
review_assignment = pd.DataFrame(review_assignment)                                                                                                                     # lista vira tabela

print("Atividades sorteadas:", len(review_assignment))
print("Atividades por pessoa:", {reviewer: len(ids_of_each_reviewer[reviewer]) for reviewer in reviewers})
print("Rotuladas por uma pessoa:", (review_assignment["number_of_reviewers"] == 1).sum(), "| por duas:", (review_assignment["number_of_reviewers"] == 2).sum())
review_assignment["reviewers"].value_counts()

Atividades sorteadas: 100
Atividades por pessoa: {'Beatriz': 30, 'Bruna': 30, 'Henrique': 30, 'João': 30, 'Júlia': 30}
Rotuladas por uma pessoa: 50 | por duas: 50


reviewers
Beatriz             10
Bruna               10
Henrique            10
João                10
Júlia               10
Beatriz | Bruna     10
Bruna | Henrique    10
Henrique | João     10
João | Júlia        10
Beatriz | Júlia     10
Name: count, dtype: int64

In [4]:
# ----------------------------------------------------------------------------------------------------
#                                         COMO FICOU A AMOSTRA
# ----------------------------------------------------------------------------------------------------
sampled_activities = activities_multilabel.set_index("activity_id").loc[sampled_ids]   # as 100 atividades, na ordem do sorteio

print("Categorias que a régua deu às atividades da amostra:", sampled_activities["number_of_labels"].value_counts().sort_index().to_dict())
print("De que instituição:", sampled_activities["source"].value_counts().to_dict())
sampled_activities["category"].value_counts()

Categorias que a régua deu às atividades da amostra: {1: 49, 2: 37, 3: 14}
De que instituição: {'SESC': 79, 'Fábricas de Cultura': 21}


category
Esporte                              31
Shows, Espetáculos e Performances    16
Sociedade e Debates                   9
Artes Visuais                         8
Cinema                                6
Infância e Brincadeiras               6
Literatura e Leitura                  6
Atividade Física e Bem-Estar          4
Música                                4
Artesanato                            4
Reciclagem e Sustentabilidade         3
Cultura e Identidade                  2
Plantas e Alimentação                 1
Name: count, dtype: int64

## O critério e o texto de cada categoria

Todos nós usamos o mesmo critério, que saiu da leitura do experimento 5 (notebook 06) e que deixamos
mais explícito aqui. A 1ª categoria é a que mais se aproxima de descrever o que a atividade é, e as
categorias seguintes entram por um de dois motivos: porque também descrevem o que a atividade é, ou
porque quem procura aquela categoria provavelmente se interessaria pela atividade.

Também escrevemos um texto curto para cada categoria, a partir das palavras e dos títulos de cada grupo,
para ninguém depender só do nome.

In [5]:
# ====================================================================================================
#                                   O TEXTO CURTO DE CADA CATEGORIA
# ====================================================================================================
description_of_each_category = {
    "Artes Visuais": "desenho, pintura, fotografia, gravura, design, vídeo e outras formas de criar imagens.",
    "Artesanato": "fazer peças com as mãos: costura, crochê, tecelagem, cerâmica, marcenaria, bijuteria.",
    "Atividade Física e Bem-Estar": "cuidar do corpo sem competir: yoga, pilates, ginástica, alongamento, meditação, dança como exercício.",
    "Cinema": "exibição de filmes, mostras, cineclubes e atividades sobre cinema.",
    "Cultura e Identidade": "atividades centradas na cultura e na identidade de um povo ou grupo: negra, indígena, periférica, nordestina, entre outras.",
    "Esporte": "modalidades esportivas, em aula, treino, torneio ou recreação: futebol, vôlei, basquete, natação, corrida, lutas, skate, tênis de mesa.",
    "Infância e Brincadeiras": "atividades feitas para crianças e bebês, com ou sem a família: brincar, jogos, espaços de brincar.",
    "Literatura e Leitura": "livros, leitura, escrita, poesia, contação de histórias, clubes de leitura, encontros com autores.",
    "Música": "aprender e praticar música: aulas de instrumento e de canto, orquestra, coral, prática em grupo.",
    "Plantas e Alimentação": "plantas, horta, permacultura, ervas medicinais, culinária, feiras de alimentos.",
    "Reciclagem e Sustentabilidade": "meio ambiente, reciclagem, lixo, consumo consciente, clima.",
    "Shows, Espetáculos e Performances": "apresentações para assistir: show de música, teatro, circo, dança, mágica, festival com palco.",
    "Sociedade e Debates": "palestras, cursos, seminários e rodas de conversa sobre temas da sociedade: educação, história, política, direitos, tecnologia, saúde mental.",
}   # categoria -> o que entra nela

category_names = sorted(description_of_each_category)                                                                                # as 13 categorias, em ordem alfabética
number_of_each_category = {category_name: category_position + 1 for category_position, category_name in enumerate(category_names)}   # número que a pessoa escreve na planilha

list_of_categories = ""                                                                                                                                    # a lista numerada que vai nas instruções
for category_name in category_names:
    list_of_categories += str(number_of_each_category[category_name]) + ". " + category_name + ": " + description_of_each_category[category_name] + "\n"   # uma linha por categoria

criterion = "**1ª categoria:** a que mais se aproxima de descrever o que a atividade é.\n\n"
criterion += "**Categorias seguintes:** entram por um de dois motivos. Ou a categoria também descreve o que a atividade é, ou quem procura essa categoria provavelmente se interessaria por esta atividade.\n\n"

print(criterion)
print(list_of_categories)

**1ª categoria:** a que mais se aproxima de descrever o que a atividade é.

**Categorias seguintes:** entram por um de dois motivos. Ou a categoria também descreve o que a atividade é, ou quem procura essa categoria provavelmente se interessaria por esta atividade.


1. Artes Visuais: desenho, pintura, fotografia, gravura, design, vídeo e outras formas de criar imagens.
2. Artesanato: fazer peças com as mãos: costura, crochê, tecelagem, cerâmica, marcenaria, bijuteria.
3. Atividade Física e Bem-Estar: cuidar do corpo sem competir: yoga, pilates, ginástica, alongamento, meditação, dança como exercício.
4. Cinema: exibição de filmes, mostras, cineclubes e atividades sobre cinema.
5. Cultura e Identidade: atividades centradas na cultura e na identidade de um povo ou grupo: negra, indígena, periférica, nordestina, entre outras.
6. Esporte: modalidades esportivas, em aula, treino, torneio ou recreação: futebol, vôlei, basquete, natação, corrida, lutas, skate, tênis de mesa.
7. Infância e Br

In [6]:
# ====================================================================================================
#                                    UMA PLANILHA PARA CADA PESSOA
# ====================================================================================================
file_name_of_each_reviewer = {"Beatriz": "beatriz", "Bruna": "bruna", "Henrique": "henrique", "João": "joao", "Júlia": "julia"}   # nome sem acento, para o nome do arquivo
maximum_labels = 4                                                                                                                # quantas categorias a pessoa pode dar a uma atividade

for reviewer in reviewers:
    activities_of_the_reviewer = sampled_activities.loc[ids_of_each_reviewer[reviewer]]                                                            # as 30 atividades dela, já embaralhadas
    sheet = pd.DataFrame({"id": activities_of_the_reviewer.index})                                                                                 # a planilha começa pelo identificador
    sheet["titulo"] = activities_of_the_reviewer["title"].values                                                                                   # título da atividade
    sheet["descricao"] = activities_of_the_reviewer["description"].values                                                                          # descrição completa
    for label_position in range(maximum_labels):
        sheet["categoria_" + str(label_position + 1)] = ""                                                                                         # a pessoa escreve o número da categoria, da mais importante para a menos
    sheet["observacao"] = ""                                                                                                                       # opcional, para dúvida ou para quando nenhuma categoria serve
    answer_columns = [column for column in sheet.columns if column not in ["id", "titulo", "descricao"]]                                           # as colunas que a pessoa preenche
    sheet = sheet[answer_columns + ["id", "titulo", "descricao"]]                                                                                  # respostas à esquerda e o texto longo por último, para ele não cobrir as outras colunas no Google Sheets
    sheet.to_csv("../data/04_review/01_sent/sheet_" + file_name_of_each_reviewer[reviewer] + ".csv", sep=",", index=False, encoding="utf-8-sig")   # separado por vírgula, que é como o Google Sheets lê e devolve um CSV
    print("Planilha salva: data/04_review/01_sent/sheet_" + file_name_of_each_reviewer[reviewer] + ".csv |", len(sheet), "atividades")

review_assignment.to_csv("../data/04_review/01_sent/review_assignment.csv", sep=";", index=False, encoding="utf-8-sig")   # quem rotula cada atividade, para a leitura das respostas
sheet.head(3)

Planilha salva: data/04_review/01_sent/sheet_beatriz.csv | 30 atividades
Planilha salva: data/04_review/01_sent/sheet_bruna.csv | 30 atividades
Planilha salva: data/04_review/01_sent/sheet_henrique.csv | 30 atividades
Planilha salva: data/04_review/01_sent/sheet_joao.csv | 30 atividades
Planilha salva: data/04_review/01_sent/sheet_julia.csv | 30 atividades


,categoria_1,categoria_2,categoria_3,categoria_4,observacao,id,titulo,descricao
0,,,,,,120,HISTÓRIAS PRETINHAS PARA TODAS AS IDADES,O projeto de contação de histórias “Histórias ...
1,,,,,,250,Tarde de Jogos,"Aqui a cooperação, a estratégia e a criativida..."
2,,,,,,72,DIA DO SACI | JAXY JATERÊ – CONFECÇÃO DE MARCA...,Você já ouviu falar em Jaxy Jaterê? Nesta ofic...


In [7]:
# ----------------------------------------------------------------------------------------------------
#                             AS INSTRUÇÕES QUE VÃO JUNTO COM A PLANILHA
# ----------------------------------------------------------------------------------------------------
instructions = "# Rotulagem das atividades: como preencher a planilha\n\n"
instructions += "Você recebeu uma planilha com 30 atividades do SESC e das Fábricas de Cultura. Para cada uma, escolha na lista abaixo as categorias que combinam com ela.\n\n"
instructions += "## O critério\n\n"
instructions += criterion
instructions += "## O que preencher\n\n"
instructions += "- **categoria_1**: o número da categoria que mais se aproxima de descrever o que a atividade é.\n"
instructions += "- **categoria_2, categoria_3, categoria_4**: os números das categorias seguintes, da mais importante para a menos. Preencha só as que fizerem sentido. Se a atividade tem uma categoria só, deixe as outras vazias.\n"
instructions += "- **observacao**: opcional. Use se ficou em dúvida, se não deu para julgar pela descrição ou se nenhuma das 13 categorias serve (nesse caso, deixe a categoria_1 vazia).\n\n"
instructions += "## Cuidados\n\n"
instructions += "- Leia o título e a descrição inteira antes de responder.\n"
instructions += "- Não existe quantidade certa de categorias. Uma atividade pode ter uma só.\n"
instructions += "- Responda sem combinar com ninguém e sem mostrar a sua planilha. Parte das atividades também está na planilha de outra pessoa, e é comparando as duas respostas que medimos se o critério está claro.\n"
instructions += "- Não mude as outras colunas nem a ordem das linhas.\n"
instructions += "\n## Como abrir e devolver\n\n"
instructions += "- Preencha no Google Sheets. Envie o arquivo para o seu Google Drive e escolha Abrir com > Planilhas Google. Use a sua conta e não compartilhe a planilha com ninguém do grupo.\n"
instructions += "- Ao terminar, baixe em Arquivo > Fazer download > Valores separados por vírgula (.csv) e envie esse arquivo para a Bruna.\n\n"
instructions += "## As 13 categorias\n\n"
instructions += list_of_categories

instructions_file = open("../data/04_review/01_sent/instructions.md", "w", encoding="utf-8")   # arquivo de texto que acompanha as planilhas
instructions_file.write(instructions)                                                          # grava o texto
instructions_file.close()                                                                      # fecha o arquivo

print(instructions)

# Rotulagem das atividades: como preencher a planilha

Você recebeu uma planilha com 30 atividades do SESC e das Fábricas de Cultura. Para cada uma, escolha na lista abaixo as categorias que combinam com ela.

## O critério

**1ª categoria:** a que mais se aproxima de descrever o que a atividade é.

**Categorias seguintes:** entram por um de dois motivos. Ou a categoria também descreve o que a atividade é, ou quem procura essa categoria provavelmente se interessaria por esta atividade.

## O que preencher

- **categoria_1**: o número da categoria que mais se aproxima de descrever o que a atividade é.
- **categoria_2, categoria_3, categoria_4**: os números das categorias seguintes, da mais importante para a menos. Preencha só as que fizerem sentido. Se a atividade tem uma categoria só, deixe as outras vazias.
- **observacao**: opcional. Use se ficou em dúvida, se não deu para julgar pela descrição ou se nenhuma das 13 categorias serve (nesse caso, deixe a categoria_1 vazia).

## Cuidado

## O arquivo para o julgador final

Depois de nós cinco, um modelo de linguagem rotula as mesmas 100 atividades do zero, com o mesmo
critério e os mesmos textos das categorias, numa conversa nova e sem nenhuma informação do projeto. Ele
não vê as categorias da régua nem as nossas respostas, e não entra no gabarito, servindo para medirmos o
quanto um julgador automático concorda conosco. O arquivo abaixo é tudo o que ele recebe.

In [8]:
# ====================================================================================================
#                               ARQUIVO ÚNICO PARA O MODELO DE LINGUAGEM
# ====================================================================================================
llm_input = "Você vai classificar 100 atividades culturais e educacionais em categorias. Para cada atividade, escolha na lista abaixo as categorias que combinam com ela.\n\n"
llm_input += "Critério:\n\n" + criterion.replace("**", "")   # o mesmo critério das pessoas, sem a marcação de negrito
llm_input += "Regras:\n"
llm_input += "- Use o número da categoria, da lista abaixo.\n"
llm_input += "- Dê de 1 a 4 categorias, da mais importante para a menos. Não existe quantidade certa: uma atividade pode ter uma só.\n"
llm_input += "- Se nenhuma das 13 categorias servir, ou se não der para julgar pela descrição, deixe categoria_1 vazia e explique em observacao.\n\n"
llm_input += "As 13 categorias:\n" + list_of_categories + "\n"
llm_input += "Responda só com uma tabela em texto, sem explicação antes ou depois: a linha de cabeçalho abaixo e uma linha por atividade, com as colunas separadas por ponto e vírgula.\n"
llm_input += "id;categoria_1;categoria_2;categoria_3;categoria_4;observacao\n\n"

for activity_id in sorted(sampled_ids):
    activity = sampled_activities.loc[activity_id]   # a linha da atividade
    llm_input += "=" * 60 + "\n"
    llm_input += "id: " + str(activity_id) + "\n"
    llm_input += "Título: " + activity["title"] + "\n"
    llm_input += "Descrição: " + activity["description"] + "\n\n"

llm_file = open("../data/04_review/01_sent/llm_judge_input.txt", "w", encoding="utf-8")   # o arquivo que a instância nova recebe
llm_file.write(llm_input)                                                                 # grava o texto
llm_file.close()                                                                          # fecha o arquivo

print("Arquivo salvo em data/04_review/01_sent/llm_judge_input.txt |", len(llm_input), "caracteres")
print(llm_input[:2600])

Arquivo salvo em data/04_review/01_sent/llm_judge_input.txt | 87086 caracteres
Você vai classificar 100 atividades culturais e educacionais em categorias. Para cada atividade, escolha na lista abaixo as categorias que combinam com ela.

Critério:

1ª categoria: a que mais se aproxima de descrever o que a atividade é.

Categorias seguintes: entram por um de dois motivos. Ou a categoria também descreve o que a atividade é, ou quem procura essa categoria provavelmente se interessaria por esta atividade.

Regras:
- Use o número da categoria, da lista abaixo.
- Dê de 1 a 4 categorias, da mais importante para a menos. Não existe quantidade certa: uma atividade pode ter uma só.
- Se nenhuma das 13 categorias servir, ou se não der para julgar pela descrição, deixe categoria_1 vazia e explique em observacao.

As 13 categorias:
1. Artes Visuais: desenho, pintura, fotografia, gravura, design, vídeo e outras formas de criar imagens.
2. Artesanato: fazer peças com as mãos: costura, crochê, tecelage

## As respostas da roda

Preenchemos as planilhas no Google Sheets e baixamos cada uma como CSV em `data/04_review/02_answers/`,
e elas ficam como chegaram, tratadas como dado bruto, sem nenhuma resposta corrigida à mão. Dois de nós
escreveram o nome das categorias em vez do número, então, em vez de mexer nas planilhas, a leitura
aceita os dois jeitos e mostra cada conversão que fez, para dar para conferir.

In [9]:
# ====================================================================================================
#                                     LEITURA DAS CINCO PLANILHAS
# ====================================================================================================
import re            # reconhece se a célula é um número
import unicodedata   # tira os acentos para comparar nomes escritos de jeitos diferentes

answer_files = {"Beatriz": "../data/04_review/02_answers/sheet_beatriz.csv", "Bruna": "../data/04_review/02_answers/sheet_bruna.csv", "Henrique": "../data/04_review/02_answers/sheet_henrique.csv", "João": "../data/04_review/02_answers/sheet_joao.csv", "Júlia": "../data/04_review/02_answers/sheet_julia.csv"}       # pessoa -> planilha que ela devolveu
label_columns = ["categoria_1", "categoria_2", "categoria_3", "categoria_4"]                                                                                                                                                                                                                                               # as colunas que a pessoa preencheu, em ordem

raw_answers = {}                                                                                                          # pessoa -> a planilha dela, do jeito que chegou
for reviewer, answer_file in answer_files.items():
    answers_of_the_reviewer = pd.read_csv(answer_file, sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)   # o Google Sheets devolve separado por vírgula; tudo como texto, para nada ser convertido sem a gente ver
    answers_of_the_reviewer = answers_of_the_reviewer[answers_of_the_reviewer["id"].str.strip() != ""]                    # tira linhas vazias no fim da planilha, se houver
    answers_of_the_reviewer = answers_of_the_reviewer.reindex(columns=label_columns + ["observacao", "id"])               # coluna que ninguém preencheu volta vazia
    answers_of_the_reviewer = answers_of_the_reviewer.fillna("")                                                          # célula vazia vira texto vazio
    answers_of_the_reviewer["id"] = answers_of_the_reviewer["id"].astype(int)                                             # o identificador volta a ser número
    expected_ids = set(ids_of_each_reviewer[reviewer])                                                                    # as 30 atividades que essa pessoa recebeu
    raw_answers[reviewer] = answers_of_the_reviewer
    print(reviewer, "| linhas:", len(answers_of_the_reviewer), "| são exatamente as atividades que ela recebeu:", set(answers_of_the_reviewer["id"]) == expected_ids)

Beatriz | linhas: 30 | são exatamente as atividades que ela recebeu: True
Bruna | linhas: 30 | são exatamente as atividades que ela recebeu: True
Henrique | linhas: 30 | são exatamente as atividades que ela recebeu: True
João | linhas: 30 | são exatamente as atividades que ela recebeu: True
Júlia | linhas: 30 | são exatamente as atividades que ela recebeu: True


In [10]:
# ----------------------------------------------------------------------------------------------------
#                       NÚMERO OU NOME: AS DUAS FORMAS VIRAM O NOME DA CATEGORIA
# ----------------------------------------------------------------------------------------------------
def simplify(text):
    without_accents = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()   # "Infância" vira "Infancia"
    return re.sub(r"[^a-z0-9]", "", without_accents.lower())                                   # só letras e números: "Bem Estar " e "Bem-Estar" ficam iguais

category_by_number = {str(number): category_name for category_name, number in number_of_each_category.items()}   # "6" -> "Esporte"
category_by_simplified_name = {simplify(category_name): category_name for category_name in category_names}       # "esporte" -> "Esporte"

def read_category(cell):
    text = cell.strip()                                                        # tira espaços nas pontas
    if re.fullmatch(r"\d+(\.0)?", text):
        return category_by_number.get(str(int(float(text))), None), "número"   # 6.0 vira "Esporte"; número fora da lista vira None
    exact_name = text in category_names                                        # a pessoa escreveu o nome exatamente como na lista?
    return category_by_simplified_name.get(simplify(text), None), "nome igual à lista" if exact_name else "nome parecido"

human_labels = []                                                                                                                                                                                                                                          # uma linha por pessoa e atividade
conversions = []                                                                                                                                                                                                                                           # uma linha por célula preenchida, para conferir a leitura
for reviewer, answers_of_the_reviewer in raw_answers.items():
    for _, answer in answers_of_the_reviewer.iterrows():
        labels_of_the_answer = []                                                                                                                                                                                                                          # as categorias que a pessoa deu, em ordem
        problems = []                                                                                                                                                                                                                                      # o que não deu para ler
        for label_column in label_columns:
            cell = answer[label_column]                                                                                                                                                                                                                    # o que está escrito na célula
            if cell.strip() == "":
                continue
            category_name, written_as = read_category(cell)                                                                                                                                                                                                # traduz para o nome da categoria
            conversions.append({"reviewer": reviewer, "activity_id": answer["id"], "column": label_column, "written": cell, "written_as": written_as, "category": category_name})                                                                          # guarda a conversão
            if category_name is None:
                problems.append(label_column + " ilegível: " + cell)                                                                                                                                                                                       # não existe categoria com esse número ou nome
            elif category_name in labels_of_the_answer:
                problems.append(label_column + " repetida: " + category_name)                                                                                                                                                                              # a mesma categoria duas vezes
            else:
                labels_of_the_answer.append(category_name)                                                                                                                                                                                                 # entra na lista, na ordem em que a pessoa escreveu
        if answer["categoria_1"].strip() == "" and labels_of_the_answer:
            problems.append("categoria_1 vazia, mas há categorias nas colunas seguintes")                                                                                                                                                                  # pulou a primeira coluna
        human_labels.append({"reviewer": reviewer, "activity_id": answer["id"], "labels": " | ".join(labels_of_the_answer), "number_of_labels": len(labels_of_the_answer), "observacao": answer["observacao"].strip(), "problems": "; ".join(problems)})   # a resposta já traduzida
human_labels = pd.DataFrame(human_labels)                                                                                                                                                                                                                  # lista vira tabela
conversions = pd.DataFrame(conversions)                                                                                                                                                                                                                    # lista vira tabela

pd.crosstab(conversions["reviewer"], conversions["written_as"])

written_as,nome igual à lista,nome parecido,número
reviewer,,,
Beatriz,0,0,65
Bruna,0,0,50
Henrique,49,0,0
João,48,3,0
Júlia,0,0,66


In [11]:
# ----------------------------------------------------------------------------------------------------
#                           OS NOMES QUE NÃO ESTAVAM ESCRITOS IGUAL À LISTA
# ----------------------------------------------------------------------------------------------------
approximate_names = conversions[conversions["written_as"] == "nome parecido"]   # nomes que só bateram depois de tirar acento, hífen e espaço
approximate_names[["reviewer", "activity_id", "column", "written", "category"]]

,reviewer,activity_id,column,written,category
173,João,627,categoria_1,Atividade Física e Bem Estar,Atividade Física e Bem-Estar
184,João,158,categoria_1,Atividade Física e Bem Estar,Atividade Física e Bem-Estar
186,João,350,categoria_1,Atividade Física e Bem Estar,Atividade Física e Bem-Estar


In [12]:
# ----------------------------------------------------------------------------------------------------
#                                       O QUE PRECISA DE ATENÇÃO
# ----------------------------------------------------------------------------------------------------
answers_with_problems = human_labels[human_labels["problems"] != ""]           # alguma célula não deu para ler
answers_without_labels = human_labels[human_labels["number_of_labels"] == 0]   # a pessoa não deu nenhuma categoria
answers_with_observation = human_labels[human_labels["observacao"] != ""]      # a pessoa deixou um comentário

print("Respostas lidas:", len(human_labels), "| com problema:", len(answers_with_problems), "| sem nenhuma categoria:", len(answers_without_labels), "| com observação:", len(answers_with_observation))
pd.concat([answers_with_problems, answers_without_labels, answers_with_observation]).drop_duplicates(subset=["reviewer", "activity_id"])[["reviewer", "activity_id", "labels", "observacao", "problems"]]

Respostas lidas: 150 | com problema: 1 | sem nenhuma categoria: 1 | com observação: 3


,reviewer,activity_id,labels,observacao,problems
129,Júlia,571,Cinema,,categoria_2 ilegível: 18
80,Henrique,456,,Nenhum se enquadra,
26,Beatriz,549,Sociedade e Debates,Não entendi a descrição,
57,Bruna,456,Infância e Brincadeiras,"é, esse ta sem categoria. É tipo uma reunião/ ...",


Três respostas pediram uma decisão depois da leitura. Numa delas a célula tem um número que não existe
na lista, e nas outras duas quem respondeu comentou que a atividade não cabe em nenhuma categoria (ou
que não entendeu a descrição), então decidimos deixar essas respostas sem categoria.

A tabela guarda as duas versões. A coluna `labels` é a resposta como foi dada, e é com ela que medimos o
quanto concordamos entre nós, porque se usássemos a versão ajustada, uma resposta mudada depois de ver a
do colega deixaria a concordância maior do que foi. A coluna `labels_adjusted` tem os ajustes e é a que
entra no gabarito.

In [13]:
# ----------------------------------------------------------------------------------------------------
#                                 AJUSTES COMBINADOS DEPOIS DA LEITURA
# ----------------------------------------------------------------------------------------------------
adjustments = [
    {"reviewer": "Júlia", "activity_id": 571, "labels_adjusted": "Cinema", "reason": "categoria_2 escrita como 18, que não existe na lista; ficou só a 1ª categoria"},
    {"reviewer": "Bruna", "activity_id": 456, "labels_adjusted": "", "reason": "a própria pessoa comentou que nenhuma categoria serve"},
    {"reviewer": "Beatriz", "activity_id": 549, "labels_adjusted": "", "reason": "a pessoa comentou que não entendeu a descrição; ficou sem categoria"},
]   # cada ajuste com o motivo, para ficar registrado

human_labels["labels_adjusted"] = human_labels["labels"]                                                                                # começa igual à resposta original
human_labels["adjustment"] = ""                                                                                                         # motivo do ajuste, quando houver
for adjustment in adjustments:
    is_the_answer = (human_labels["reviewer"] == adjustment["reviewer"]) & (human_labels["activity_id"] == adjustment["activity_id"])   # a linha dessa pessoa e dessa atividade
    human_labels.loc[is_the_answer, "labels_adjusted"] = adjustment["labels_adjusted"]                                                  # troca só a versão ajustada
    human_labels.loc[is_the_answer, "adjustment"] = adjustment["reason"]                                                                # guarda o motivo

human_labels[human_labels["adjustment"] != ""][["reviewer", "activity_id", "labels", "labels_adjusted", "adjustment"]]

,reviewer,activity_id,labels,labels_adjusted,adjustment
26,Beatriz,549,Sociedade e Debates,,a pessoa comentou que não entendeu a descrição...
57,Bruna,456,Infância e Brincadeiras,,a própria pessoa comentou que nenhuma categori...
129,Júlia,571,Cinema,Cinema,"categoria_2 escrita como 18, que não existe na..."


In [14]:
# ----------------------------------------------------------------------------------------------------
#                                 A TABELA DAS RESPOSTAS, JÁ TRADUZIDA
# ----------------------------------------------------------------------------------------------------
human_labels.to_csv("../data/04_review/03_results/human_labels.csv", sep=";", index=False, encoding="utf-8-sig")   # uma linha por pessoa e atividade, com as categorias pelo nome

print("Tabela salva em data/04_review/03_results/human_labels.csv:", human_labels.shape)
print("Categorias por resposta, como as pessoas deram:", human_labels["number_of_labels"].value_counts().sort_index().to_dict())
human_labels.head(5)

Tabela salva em data/04_review/03_results/human_labels.csv: (150, 8)
Categorias por resposta, como as pessoas deram: {0: 1, 1: 54, 2: 68, 3: 18, 4: 9}


,reviewer,activity_id,labels,number_of_labels,observacao,problems,labels_adjusted,adjustment
0,Beatriz,227,Esporte | Cultura e Identidade | Atividade Fís...,4,,,Esporte | Cultura e Identidade | Atividade Fís...,
1,Beatriz,109,"Shows, Espetáculos e Performances | Música",2,,,"Shows, Espetáculos e Performances | Música",
2,Beatriz,235,Esporte | Atividade Física e Bem-Estar,2,,,Esporte | Atividade Física e Bem-Estar,
3,Beatriz,63,Sociedade e Debates,1,,,Sociedade e Debates,
4,Beatriz,495,Literatura e Leitura | Música,2,,,Literatura e Leitura | Música,


## Onde discordamos

Nas 50 atividades em dobro, comparamos as duas respostas já com os ajustes. Não contamos como
discordância quando os dois demos a mesma 1ª categoria e todas as categorias de um estão na lista do
outro, porque nesse caso um de nós só foi mais fundo e acrescentou categorias seguintes. O resto
separamos por tipo, da mais leve para a mais forte: as mesmas categorias em outra ordem; a mesma 1ª
categoria, mas cada um com uma categoria seguinte que o outro não deu; uma categoria em comum, mas a 1ª
diferente; e nenhuma categoria em comum. A lista vai para `data/04_review/03_results/disagreements.csv`,
para decidirmos como cada uma entra no gabarito.

In [15]:
# ====================================================================================================
#                                AS 50 ATIVIDADES EM DOBRO, LADO A LADO
# ====================================================================================================
def compare_answers(first_labels, second_labels):
    first_list = [label for label in first_labels.split(" | ") if label != ""]                            # as categorias da 1ª pessoa, em ordem
    second_list = [label for label in second_labels.split(" | ") if label != ""]                          # as da 2ª pessoa
    if first_list == second_list:
        return "iguais"
    same_first = first_list != [] and second_list != [] and first_list[0] == second_list[0]               # as duas dizem que a atividade é a mesma coisa
    one_contains_the_other = set(first_list) <= set(second_list) or set(second_list) <= set(first_list)   # todas as categorias de uma estão na outra
    if same_first and one_contains_the_other:
        return "concordam: uma pessoa foi mais fundo"
    if set(first_list) == set(second_list):
        return "1. mesmas categorias, outra ordem"
    if same_first:
        return "2. mesma 1ª categoria, cada uma com uma seguinte que a outra não deu"
    if set(first_list) & set(second_list):
        return "3. alguma categoria em comum, 1ª diferente"
    return "4. nenhuma categoria em comum"

double_ids = review_assignment.loc[review_assignment["number_of_reviewers"] == 2, "activity_id"]   # as 50 atividades vistas por duas pessoas
answers_by_activity = human_labels.set_index(["activity_id", "reviewer"])                          # para achar a resposta de uma pessoa numa atividade

pairs = []                                                                                                                                                                                                                                                                           # uma linha por atividade em dobro
for activity_id in double_ids:
    first_reviewer, second_reviewer = review_assignment.loc[review_assignment["activity_id"] == activity_id, "reviewers"].iloc[0].split(" | ")                                                                                                                                       # as duas pessoas, em ordem alfabética
    first_labels = answers_by_activity.loc[(activity_id, first_reviewer), "labels_adjusted"]                                                                                                                                                                                         # resposta da 1ª pessoa, com os ajustes
    second_labels = answers_by_activity.loc[(activity_id, second_reviewer), "labels_adjusted"]                                                                                                                                                                                       # resposta da 2ª pessoa
    pairs.append({"activity_id": activity_id, "title": sampled_activities.loc[activity_id, "title"], "reviewer_a": first_reviewer, "labels_a": first_labels, "reviewer_b": second_reviewer, "labels_b": second_labels, "situation": compare_answers(first_labels, second_labels)})   # as duas respostas lado a lado
pairs = pd.DataFrame(pairs).sort_values(["situation", "activity_id"])                                                                                                                                                                                                                # as mais fortes no fim

pairs["situation"].value_counts().sort_index()

situation
1. mesmas categorias, outra ordem                                        2
2. mesma 1ª categoria, cada uma com uma seguinte que a outra não deu     2
3. alguma categoria em comum, 1ª diferente                               8
4. nenhuma categoria em comum                                            4
concordam: uma pessoa foi mais fundo                                    21
iguais                                                                  13
Name: count, dtype: int64

In [16]:
# ----------------------------------------------------------------------------------------------------
#                                      A LISTA DAS DISCORDÂNCIAS
# ----------------------------------------------------------------------------------------------------
agreement_situations = ["iguais", "concordam: uma pessoa foi mais fundo"]                                            # o que não conta como discordância
disagreements = pairs[~pairs["situation"].isin(agreement_situations)]                                                # só onde as duas respostas discordam de verdade
disagreements.to_csv("../data/04_review/03_results/disagreements.csv", sep=";", index=False, encoding="utf-8-sig")   # lista para decidir como cada uma entra no gabarito

print("Atividades em dobro:", len(pairs), "| concordam:", len(pairs) - len(disagreements), "| discordam:", len(disagreements))
print("Lista salva em data/04_review/03_results/disagreements.csv")
disagreements

Atividades em dobro: 50 | concordam: 34 | discordam: 16
Lista salva em data/04_review/03_results/disagreements.csv


,activity_id,title,reviewer_a,labels_a,reviewer_b,labels_b,situation
10,299,Queimada paintball,Bruna,Infância e Brincadeiras | Esporte,Henrique,Esporte | Infância e Brincadeiras,"1. mesmas categorias, outra ordem"
36,350,Práticas Aquáticas | Aprendendo a nadar,João,Atividade Física e Bem-Estar | Esporte,Júlia,Esporte | Atividade Física e Bem-Estar,"1. mesmas categorias, outra ordem"
40,182,Jogando com a Seleção Brasileira Feminina de V...,Beatriz,Esporte | Atividade Física e Bem-Estar | Socie...,Júlia,Esporte | Atividade Física e Bem-Estar | Cultu...,"2. mesma 1ª categoria, cada uma com uma seguin..."
26,338,Esporte Adulto - Basquete Feminino,Henrique,Esporte | Cultura e Identidade | Atividade Fís...,João,Esporte | Sociedade e Debates,"2. mesma 1ª categoria, cada uma com uma seguin..."
19,56,ESCREVENDO MINHA CANÇÃO – COM EQUIPE BIBLIOTECA,Bruna,Música,Henrique,Literatura e Leitura | Música,"3. alguma categoria em comum, 1ª diferente"
27,70,ANTES DA ANIMAÇÃO: BRINQUEDOS ÓPTICOS – COM EQ...,Henrique,Artes Visuais,João,Artesanato | Artes Visuais,"3. alguma categoria em comum, 1ª diferente"
33,72,DIA DO SACI | JAXY JATERÊ – CONFECÇÃO DE MARCA...,João,Cultura e Identidade | Artesanato,Júlia,Artesanato | Literatura e Leitura | Artes Visu...,"3. alguma categoria em comum, 1ª diferente"
41,120,HISTÓRIAS PRETINHAS PARA TODAS AS IDADES,Beatriz,Literatura e Leitura | Sociedade e Debates | C...,Júlia,"Cultura e Identidade | Shows, Espetáculos e Pe...","3. alguma categoria em comum, 1ª diferente"
37,256,Irmãos & Brothers,João,"Shows, Espetáculos e Performances | Música | S...",Júlia,"Música | Shows, Espetáculos e Performances | C...","3. alguma categoria em comum, 1ª diferente"
6,277,Stop Motion no Tempo e no Espaço,Beatriz,Artes Visuais | Cinema,Bruna,Sociedade e Debates | Cinema | Artes Visuais |...,"3. alguma categoria em comum, 1ª diferente"


## O gabarito humano

Lendo as 16 discordâncias, a maior parte pareceu válida, com a atividade ficando entre duas categorias e
cada um de nós pondo uma delas em 1º lugar, ou com um indo mais fundo que o outro. Por isso o gabarito
de uma atividade vista por dois de nós junta as duas listas, e quando as duas 1ª categorias são
diferentes, as duas valem como 1ª, enquanto a atividade vista por um só fica com a lista de quem a leu.

Quatro atividades pediram decisão depois de ler a lista. Na 135 (bolsa de crochê) e na 464 (aula de
jogos de lutas), algumas categorias não tinham apoio nenhum na descrição e pareceram deslize, então
saíram. A 367 é um curso de formação de atores, e nenhuma das 13 categorias serve para ela. A 549 também
ficou sem categoria, porque a descrição é tão vaga que nem os dois que a leram concordaram, e é o tipo
de atividade que precisaria de revisão humana.

Depois relemos o gabarito inteiro, atividade por atividade, com a descrição ao lado. Em algumas
atividades a junção trouxe uma categoria que a descrição não sustenta (quase sempre Sociedade e Debates
como categoria seguinte), e algumas respostas dadas por um só de nós estavam claramente erradas, como
uma oficina de cerâmica marcada como Atividade Física. Corrigimos só o que estava errado de fato, porque
um modelo que aprendesse ou fosse medido por esses rótulos seria puxado para o lado errado. Essas
decisões valem só para o gabarito, e a nossa concordância continua sendo medida com as respostas como
foram dadas.

In [17]:
# ====================================================================================================
#                 O GABARITO: AS DUAS LISTAS JUNTAS, COM AS DECISÕES DEPOIS DA LEITURA
# ====================================================================================================
def split_labels(labels):
    return [label for label in labels.split(" | ") if label != ""]   # texto "A | B" vira a lista [A, B]

def merge_answers(answers_of_the_activity):
    first_categories = []                             # a 1ª categoria de cada pessoa (podem ser duas diferentes)
    for labels in answers_of_the_activity:
        labels_list = split_labels(labels)            # a lista de uma pessoa
        if labels_list and labels_list[0] not in first_categories:
            first_categories.append(labels_list[0])   # guarda a 1ª dela, sem repetir
    merged_labels = list(first_categories)            # o gabarito começa pelas 1ªs categorias
    for labels in answers_of_the_activity:
        for label in split_labels(labels):
            if label not in merged_labels:
                merged_labels.append(label)           # depois as seguintes, na ordem em que apareceram
    return merged_labels, first_categories

gold_decisions = {
    135: {"gold_labels": ["Artesanato"], "reason": "Sociedade e Debates saiu: a descrição é só a confecção de uma bolsa de crochê"},
    464: {"gold_labels": ["Esporte"], "reason": "Atividade Física e Sociedade e Debates saíram: o texto de Esporte cita lutas, e a descrição não fala de debate"},
    367: {"gold_labels": [], "reason": "sem categoria: é um curso de formação de atores, e nenhuma das 13 categorias serve"},
    549: {"gold_labels": [], "reason": "sem categoria: a descrição é confusa demais para decidir, é um caso para revisão humana"},
    256: {"gold_labels": ["Shows, Espetáculos e Performances", "Música", "Cultura e Identidade"], "gold_first_categories": ["Shows, Espetáculos e Performances", "Música"], "reason": "Sociedade e Debates saiu: é um show, a descrição não fala de debate"},
    311: {"gold_labels": ["Música", "Infância e Brincadeiras"], "reason": "Sociedade e Debates saiu: é uma aula de violino para crianças"},
    338: {"gold_labels": ["Esporte", "Atividade Física e Bem-Estar"], "reason": "Cultura e Identidade e Sociedade e Debates saíram: a descrição é a de um curso de esporte, só o título diz feminino"},
    688: {"gold_labels": ["Esporte"], "reason": "Sociedade e Debates saiu: a descrição é a de um curso de esporte, só o título diz feminino"},
    674: {"gold_labels": ["Artesanato"], "reason": "Sociedade e Debates saiu: é um curso de técnicas de marcenaria"},
    513: {"gold_labels": ["Plantas e Alimentação", "Sociedade e Debates", "Reciclagem e Sustentabilidade"], "reason": "Atividade Física saiu: a palestra é sobre carne cultivada e sistemas alimentares"},
    642: {"gold_labels": ["Shows, Espetáculos e Performances", "Infância e Brincadeiras"], "reason": "Cinema saiu: é um espetáculo de teatro com pernas de pau"},
    536: {"gold_labels": ["Música", "Sociedade e Debates"], "gold_first_categories": ["Música", "Sociedade e Debates"], "reason": "Literatura e Leitura saiu: é uma palestra sobre um compositor"},
    277: {"gold_labels": ["Artes Visuais", "Cinema", "Artesanato"], "gold_first_categories": ["Artes Visuais", "Cinema"], "reason": "Sociedade e Debates saiu: o curso é sobre a técnica e a história do stop motion; Cinema passa a valer como 1ª"},
    671: {"gold_labels": ["Artesanato", "Artes Visuais"], "reason": "Atividade Física virou Artesanato: é uma oficina de cerâmica"},
    69: {"gold_labels": ["Artesanato"], "reason": "Literatura e Leitura virou Artesanato: é uma oficina de costura e tecido"},
    473: {"gold_labels": ["Shows, Espetáculos e Performances", "Artes Visuais"], "reason": "a ordem foi invertida: o curso é de arte mágica, e o texto de Shows cita mágica"},
    346: {"gold_labels": ["Música"], "reason": "Sociedade e Debates saiu: é uma aula de contrabaixo"},
    39: {"gold_labels": ["Música", "Shows, Espetáculos e Performances"], "reason": "Artes Visuais virou Shows: é a apresentação de um DJ num festival"},
}   # atividade -> como ficou no gabarito depois de lermos as discordâncias e o gabarito inteiro

gold_rows = []                                                                                                                                                                                                                                                                                                                                  # uma linha por atividade sorteada
for activity_id in sampled_ids:
    reviewers_of_the_activity = review_assignment.loc[review_assignment["activity_id"] == activity_id, "reviewers"].iloc[0].split(" | ")                                                                                                                                                                                                        # uma ou duas pessoas
    answers_of_the_activity = [answers_by_activity.loc[(activity_id, reviewer), "labels_adjusted"] for reviewer in reviewers_of_the_activity]                                                                                                                                                                                                   # as respostas, já com os ajustes
    gold_labels, first_categories = merge_answers(answers_of_the_activity)                                                                                                                                                                                                                                                                      # junta as listas
    how_it_was_built = "uma pessoa" if len(reviewers_of_the_activity) == 1 else "duas pessoas, listas juntas"                                                                                                                                                                                                                                   # de onde veio o gabarito
    if activity_id in gold_decisions:
        gold_labels = gold_decisions[activity_id]["gold_labels"]                                                                                                                                                                                                                                                                                # a decisão tomada depois da leitura
        first_categories = gold_decisions[activity_id].get("gold_first_categories", gold_labels[:1])                                                                                                                                                                                                                                            # a 1ª da lista decidida, ou as duas quando as duas valem
        how_it_was_built = "decisão depois da leitura: " + gold_decisions[activity_id]["reason"]                                                                                                                                                                                                                                                # registra o motivo
    gold_rows.append({"activity_id": activity_id, "title": sampled_activities.loc[activity_id, "title"], "reviewers": " | ".join(reviewers_of_the_activity), "gold_labels": " | ".join(gold_labels), "gold_first_categories": " | ".join(first_categories), "number_of_gold_labels": len(gold_labels), "how_it_was_built": how_it_was_built})   # a linha do gabarito
gold_labels_table = pd.DataFrame(gold_rows).sort_values("activity_id")                                                                                                                                                                                                                                                                          # lista vira tabela, na ordem do identificador

gold_labels_table.to_csv("../data/04_review/03_results/gold_labels.csv", sep=";", index=False, encoding="utf-8-sig")   # o gabarito humano das 100 atividades

print("Gabarito salvo em data/04_review/03_results/gold_labels.csv:", gold_labels_table.shape)
print("Categorias por atividade:", gold_labels_table["number_of_gold_labels"].value_counts().sort_index().to_dict())
print("Atividades com duas 1ªs categorias válidas:", (gold_labels_table["gold_first_categories"].str.contains(" | ", regex=False)).sum())
print("Atividades sem categoria:", gold_labels_table.loc[gold_labels_table["number_of_gold_labels"] == 0, "activity_id"].tolist())

Gabarito salvo em data/04_review/03_results/gold_labels.csv: (100, 7)
Categorias por atividade: {0: 3, 1: 31, 2: 44, 3: 13, 4: 9}
Atividades com duas 1ªs categorias válidas: 10
Atividades sem categoria: [367, 456, 549]


In [18]:
# ----------------------------------------------------------------------------------------------------
#                           QUANTAS VEZES CADA CATEGORIA APARECE NO GABARITO
# ----------------------------------------------------------------------------------------------------
labels_in_gold = gold_labels_table["gold_labels"].str.split(" | ", regex=False).explode()            # uma linha por categoria dada
labels_in_gold = labels_in_gold[labels_in_gold != ""]                                                # tira as atividades sem categoria
first_in_gold = gold_labels_table["gold_first_categories"].str.split(" | ", regex=False).explode()   # uma linha por 1ª categoria válida
first_in_gold = first_in_gold[first_in_gold != ""]                                                   # tira as atividades sem categoria

categories_in_gold = pd.DataFrame({"in_any_position": labels_in_gold.value_counts(), "as_first": first_in_gold.value_counts()}).fillna(0).astype(int)   # uma linha por categoria
categories_in_gold.sort_values("in_any_position", ascending=False)

,in_any_position,as_first
Esporte,31,27
Atividade Física e Bem-Estar,23,9
Sociedade e Debates,21,9
Cultura e Identidade,19,3
"Shows, Espetáculos e Performances",18,13
Música,18,10
Infância e Brincadeiras,15,6
Artesanato,14,9
Artes Visuais,11,4
Literatura e Leitura,9,7


## O julgador automático

O modelo de linguagem (Claude Opus 5.5, com esforço alto) recebeu o arquivo gerado acima numa conversa
nova, em 27/10/2026, e devolveu uma tabela com as categorias de cada uma das 100 atividades. A resposta
foi salva como veio em `data/04_review/02_answers/sheet_llm.csv` e é lida do mesmo jeito que as nossas
planilhas.

In [19]:
# ====================================================================================================
#                                    LEITURA DA RESPOSTA DO MODELO
# ====================================================================================================
llm_answers = pd.read_csv("../data/04_review/02_answers/sheet_llm.csv", sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)   # a tabela que o modelo devolveu
llm_answers["id"] = llm_answers["id"].astype(int)                                                                                          # o identificador volta a ser número

llm_rows = []                                                                                                                                                                                                                # uma linha por atividade, já com o nome das categorias
for _, answer in llm_answers.iterrows():
    labels_of_the_answer = []                                                                                                                                                                                                # as categorias que o modelo deu, em ordem
    problems = []                                                                                                                                                                                                            # o que não deu para ler
    for label_column in label_columns:
        cell = answer[label_column]                                                                                                                                                                                          # o que está escrito na célula
        if cell.strip() == "":
            continue
        category_name, written_as = read_category(cell)                                                                                                                                                                      # traduz para o nome da categoria
        if category_name is None:
            problems.append(label_column + " ilegível: " + cell)                                                                                                                                                             # número que não existe
        elif category_name in labels_of_the_answer:
            problems.append(label_column + " repetida: " + category_name)                                                                                                                                                    # a mesma categoria duas vezes
        else:
            labels_of_the_answer.append(category_name)                                                                                                                                                                       # entra na lista, na ordem do modelo
    llm_rows.append({"activity_id": answer["id"], "labels": " | ".join(labels_of_the_answer), "number_of_labels": len(labels_of_the_answer), "observacao": answer["observacao"].strip(), "problems": "; ".join(problems)})   # a resposta traduzida
llm_labels = pd.DataFrame(llm_rows).set_index("activity_id")                                                                                                                                                                 # lista vira tabela, indexada pela atividade

print("Atividades na resposta:", len(llm_labels), "| são exatamente as 100 sorteadas:", set(llm_labels.index) == set(sampled_ids))
print("Com problema de leitura:", (llm_labels["problems"] != "").sum(), "| sem categoria:", (llm_labels["number_of_labels"] == 0).sum())
print("Categorias por atividade:", llm_labels["number_of_labels"].value_counts().sort_index().to_dict())
llm_labels[llm_labels["observacao"] != ""][["labels", "observacao"]]

Atividades na resposta: 100 | são exatamente as 100 sorteadas: True
Com problema de leitura: 0 | sem categoria: 0
Categorias por atividade: {1: 55, 2: 31, 3: 11, 4: 3}


,labels,observacao
activity_id,,
132,Sociedade e Debates,Mostra de objetos tecnológicos antigos. Nenhum...
367,"Shows, Espetáculos e Performances",Curso de formação em teatro. 12 é a categoria ...
456,Infância e Brincadeiras,Empréstimo de jogos de tabuleiro para jovens a...


O modelo deixou observação em três atividades (132, 367 e 456), sempre dizendo que nenhuma categoria
servia bem, mas escolheu a mais próxima mesmo assim, quando as instruções pediam para deixar a 1ª
categoria vazia nesse caso. Duas delas (367 e 456) estão entre as três que ficaram sem categoria no
gabarito, e a terceira dessas, a 549, ele classificou como Literatura e Leitura sem comentário. Lemos a
resposta como ele deu.

In [20]:
# ----------------------------------------------------------------------------------------------------
#                O MODELO CONCORDA COM AS PESSOAS TANTO QUANTO ELAS CONCORDAM ENTRE SI?
# ----------------------------------------------------------------------------------------------------
human_pairs_situations = []                                                                                                                                                               # as 50 atividades em dobro, comparando as respostas originais das duas pessoas
for activity_id in double_ids:
    first_reviewer, second_reviewer = review_assignment.loc[review_assignment["activity_id"] == activity_id, "reviewers"].iloc[0].split(" | ")                                            # as duas pessoas
    human_pairs_situations.append(compare_answers(answers_by_activity.loc[(activity_id, first_reviewer), "labels"], answers_by_activity.loc[(activity_id, second_reviewer), "labels"]))   # como as respostas originais se comparam

llm_human_situations = []                                                                                                         # as 150 respostas das pessoas, cada uma comparada com a do modelo
for _, human_answer in human_labels.iterrows():
    llm_human_situations.append(compare_answers(llm_labels.loc[human_answer["activity_id"], "labels"], human_answer["labels"]))   # modelo contra uma pessoa

agreement_table = pd.DataFrame({"pessoa x pessoa (50 pares)": pd.Series(human_pairs_situations).value_counts(), "modelo x pessoa (150 pares)": pd.Series(llm_human_situations).value_counts()}).fillna(0).astype(int).sort_index()   # as duas comparações lado a lado
agreement_table.loc["concordam (iguais ou mais fundo)"] = agreement_table.loc[[situation for situation in agreement_situations if situation in agreement_table.index]].sum()                                                         # o total que não é discordância
agreement_table

,pessoa x pessoa (50 pares),modelo x pessoa (150 pares)
"1. mesmas categorias, outra ordem",2,11
"2. mesma 1ª categoria, cada uma com uma seguinte que a outra não deu",2,9
"3. alguma categoria em comum, 1ª diferente",9,21
4. nenhuma categoria em comum,4,11
concordam: uma pessoa foi mais fundo,21,51
iguais,12,47
concordam (iguais ou mais fundo),33,98


In [21]:
# ----------------------------------------------------------------------------------------------------
#                                      O MODELO CONTRA O GABARITO
# ----------------------------------------------------------------------------------------------------
gold_by_activity = gold_labels_table.set_index("activity_id")   # o gabarito, indexado pela atividade

llm_first_is_valid = []                                                                                                                 # a 1ª categoria do modelo é uma das 1ªs válidas do gabarito?
llm_against_gold = []                                                                                                                   # como a lista do modelo se compara com a do gabarito
for activity_id in sampled_ids:
    llm_list = split_labels(llm_labels.loc[activity_id, "labels"])                                                                      # a lista do modelo
    gold_first = split_labels(gold_by_activity.loc[activity_id, "gold_first_categories"])                                               # as 1ªs válidas do gabarito
    if gold_first == []:
        llm_first_is_valid.append(llm_list == [])                                                                                       # sem categoria no gabarito: só acerta quem também não deu nenhuma
    else:
        llm_first_is_valid.append(llm_list != [] and llm_list[0] in gold_first)                                                         # acertou o que a atividade é
    llm_against_gold.append(compare_answers(llm_labels.loc[activity_id, "labels"], gold_by_activity.loc[activity_id, "gold_labels"]))   # a lista inteira

print("1ª categoria do modelo entre as 1ªs válidas do gabarito:", sum(llm_first_is_valid), "de", len(sampled_ids))
pd.Series(llm_against_gold).value_counts().sort_index()

1ª categoria do modelo entre as 1ªs válidas do gabarito: 79 de 100


1. mesmas categorias, outra ordem                                        7
2. mesma 1ª categoria, cada uma com uma seguinte que a outra não deu     5
3. alguma categoria em comum, 1ª diferente                              12
4. nenhuma categoria em comum                                            5
concordam: uma pessoa foi mais fundo                                    37
iguais                                                                  34
Name: count, dtype: int64

## A concordância descontando o acaso

As comparações acima contam quantas vezes duas respostas batem, mas parte disso aconteceria por acaso:
como Esporte é a categoria mais comum, dois de nós que chutassem Esporte toda vez também concordariam
bastante. O kappa de Cohen desconta esse acaso, valendo 1 quando a concordância é total, 0 quando é a
que se esperaria só pelo acaso, e ficando no meio no resto.

Medimos de dois jeitos. No primeiro, cada atividade vira 13 perguntas de sim ou não ("tem Esporte?",
"tem Cinema?"...), e comparamos as duas respostas pergunta por pergunta, e no segundo olhamos só a 1ª
categoria. As contas usam as respostas como foram dadas, sem os ajustes do gabarito.

In [22]:
# ====================================================================================================
#                         KAPPA ENTRE AS PESSOAS E ENTRE O MODELO E AS PESSOAS
# ====================================================================================================
from sklearn.metrics import cohen_kappa_score   # concordância entre dois julgadores, descontando o acaso

def label_matrix(list_of_labels):
    matrix = np.zeros((len(list_of_labels), len(category_names)), dtype=int)   # uma linha por resposta, uma coluna por categoria
    for row_number, labels in enumerate(list_of_labels):
        for label in split_labels(labels):
            matrix[row_number, category_names.index(label)] = 1                # 1 quando a resposta tem a categoria
    return matrix

def first_category_of(labels):
    labels_list = split_labels(labels)                          # a lista da resposta
    return labels_list[0] if labels_list else "sem categoria"   # a 1ª, ou "sem categoria" quando a resposta veio vazia

pair_rows = []                                                                                                                                                                                                                                                      # uma linha por atividade em dobro, com as duas respostas originais
for activity_id in double_ids:
    first_reviewer, second_reviewer = review_assignment.loc[review_assignment["activity_id"] == activity_id, "reviewers"].iloc[0].split(" | ")                                                                                                                      # as duas pessoas
    pair_rows.append({"activity_id": activity_id, "pair": first_reviewer + " e " + second_reviewer, "labels_a": answers_by_activity.loc[(activity_id, first_reviewer), "labels"], "labels_b": answers_by_activity.loc[(activity_id, second_reviewer), "labels"]})   # guarda o par
pair_rows = pd.DataFrame(pair_rows)                                                                                                                                                                                                                                 # lista vira tabela

def agreement_summary(labels_a, labels_b):
    matrix_a = label_matrix(labels_a).ravel()                      # todas as perguntas de sim ou não do 1º julgador, numa fila só
    matrix_b = label_matrix(labels_b).ravel()                      # as mesmas perguntas do 2º julgador
    first_a = [first_category_of(labels) for labels in labels_a]   # 1ª categoria de cada resposta do 1º
    first_b = [first_category_of(labels) for labels in labels_b]   # 1ª categoria de cada resposta do 2º
    return {"answers": len(labels_a), "yes_no_agreement_%": round(100 * (matrix_a == matrix_b).mean(), 1), "yes_no_kappa": round(cohen_kappa_score(matrix_a, matrix_b), 3), "first_category_agreement_%": round(100 * np.mean(np.array(first_a) == np.array(first_b)), 1), "first_category_kappa": round(cohen_kappa_score(first_a, first_b), 3)}

llm_labels_for_each_human_answer = [llm_labels.loc[activity_id, "labels"] for activity_id in human_labels["activity_id"]]   # a resposta do modelo para a mesma atividade de cada resposta humana

agreement_rows = {}                                                                                                                        # nome da comparação -> resumo
agreement_rows["pessoa x pessoa (50 atividades)"] = agreement_summary(pair_rows["labels_a"].tolist(), pair_rows["labels_b"].tolist())      # as 50 em dobro
agreement_rows["modelo x pessoa (150 respostas)"] = agreement_summary(llm_labels_for_each_human_answer, human_labels["labels"].tolist())   # o modelo contra cada resposta humana
pd.DataFrame(agreement_rows).T

,answers,yes_no_agreement_%,yes_no_kappa,first_category_agreement_%,first_category_kappa
pessoa x pessoa (50 atividades),50.0,91.1,0.646,70.0,0.659
modelo x pessoa (150 respostas),150.0,92.9,0.697,71.3,0.670


In [23]:
# ----------------------------------------------------------------------------------------------------
#                                           POR PAR DA RODA
# ----------------------------------------------------------------------------------------------------
agreement_by_pair = {}                                                                                                               # par -> resumo das 10 atividades em comum
for pair_name, rows_of_the_pair in pair_rows.groupby("pair"):
    agreement_by_pair[pair_name] = agreement_summary(rows_of_the_pair["labels_a"].tolist(), rows_of_the_pair["labels_b"].tolist())   # a mesma conta, só com as 10 do par
pd.DataFrame(agreement_by_pair).T

,answers,yes_no_agreement_%,yes_no_kappa,first_category_agreement_%,first_category_kappa
Beatriz e Bruna,10.0,91.5,0.694,80.0,0.770
Beatriz e Júlia,10.0,90.8,0.672,80.0,0.767
Bruna e Henrique,10.0,93.8,0.680,70.0,0.647
Henrique e João,10.0,90.0,0.549,60.0,0.481
João e Júlia,10.0,89.2,0.621,60.0,0.518


**O que deu.** Entre nós, o kappa ficou em 0,65 nas perguntas de sim ou não e em 0,66 na 1ª categoria, e
demos a mesma 1ª categoria em 70% das 50 atividades. A porcentagem de acordo nas perguntas de sim ou não
(91%) parece alta, mas é inflada pelos "não", já que a maioria das 13 categorias não se aplica a quase
nenhuma atividade e concordar nelas é fácil, e por isso olhamos o kappa.

O modelo de linguagem concordou conosco um pouco mais do que concordamos entre nós (0,70 e 0,67). Não
lemos isso como o modelo sendo melhor que nós, porque ele foi comparado com 150 respostas, e nós com 50
pares, e a diferença é pequena, então o que dá para dizer é que ele rotula num nível parecido com o
nosso.

Por par, a concordância vai de 0,55 (Henrique e João) a 0,69 (Beatriz e Bruna) nas perguntas de sim ou
não. Como foi a Bruna quem nomeou as categorias na Parte 2, olhamos se os pares dela destoavam: os dois
ficam no alto dessa faixa (0,69 e 0,68), mas o par Beatriz e Júlia, sem ela, fica quase igual (0,67),
então não aparece sinal claro de que ela tenha julgado de um jeito diferente do resto do grupo. Com 10
atividades por par, essas diferenças são frágeis.

## O que o gabarito diz da régua

O mesmo gabarito mede as duas etapas do projeto. A Parte 2 dava uma categoria por atividade, que
corresponde à 1ª categoria da régua, e para ela contamos quantas vezes essa categoria está entre as 1ªs
categorias válidas do gabarito e quantas vezes está em qualquer posição do gabarito. A Parte 3 dá a
lista inteira, e nela olhamos quantas categorias da régua estão no gabarito (precisão) e quantas
categorias do gabarito a régua deu (cobertura). O modelo de linguagem entra na tabela só como
referência, e a comparação completa, com os classificadores, fica para o notebook 09.

In [24]:
# ====================================================================================================
#                                 A RÉGUA E O MODELO CONTRA O GABARITO
# ====================================================================================================
def against_gold(predicted_by_activity):
    first_in_gold_first = 0                                                                     # 1ª categoria prevista entre as 1ªs válidas do gabarito
    first_in_gold_any = 0                                                                       # 1ª categoria prevista em qualquer posição do gabarito
    predicted_total = 0                                                                         # quantas categorias foram previstas
    gold_total = 0                                                                              # quantas categorias o gabarito tem
    correct_total = 0                                                                           # quantas previstas estão no gabarito
    for activity_id in sampled_ids:
        predicted_list = split_labels(predicted_by_activity[activity_id])                       # a lista prevista
        gold_list = split_labels(gold_by_activity.loc[activity_id, "gold_labels"])              # a lista do gabarito
        gold_first = split_labels(gold_by_activity.loc[activity_id, "gold_first_categories"])   # as 1ªs válidas
        if predicted_list and predicted_list[0] in gold_first:
            first_in_gold_first += 1                                                            # acertou o que a atividade é
        if predicted_list and predicted_list[0] in gold_list:
            first_in_gold_any += 1                                                              # a 1ª prevista está certa, mesmo que não como 1ª
        predicted_total += len(predicted_list)                                                  # soma as previstas
        gold_total += len(gold_list)                                                            # soma as do gabarito
        correct_total += len(set(predicted_list) & set(gold_list))                              # soma as que batem
    return {"first_among_gold_first": first_in_gold_first, "first_anywhere_in_gold": first_in_gold_any, "predicted_labels": predicted_total, "precision_%": round(100 * correct_total / predicted_total, 1), "coverage_%": round(100 * correct_total / gold_total, 1)}

ruler_labels = activities_multilabel.set_index("activity_id")["labels"]   # as listas da régua, do notebook 07
ruler_first_only = ruler_labels.str.split(" | ", regex=False).str[0]      # a versão da Parte 2: só a 1ª categoria

against_gold_rows = {}                                                                      # quem -> resumo
against_gold_rows["Parte 2: régua, só a 1ª categoria"] = against_gold(ruler_first_only)     # uma categoria por atividade
against_gold_rows["Parte 3: régua, lista inteira"] = against_gold(ruler_labels)             # a lista da régua
against_gold_rows["referência: modelo de linguagem"] = against_gold(llm_labels["labels"])   # o julgador automático
pd.DataFrame(against_gold_rows).T

,first_among_gold_first,first_anywhere_in_gold,predicted_labels,precision_%,coverage_%
"Parte 2: régua, só a 1ª categoria",68.0,82.0,100.0,82.0,42.3
"Parte 3: régua, lista inteira",68.0,82.0,165.0,70.3,59.8
referência: modelo de linguagem,79.0,90.0,162.0,84.6,70.6


In [25]:
# ----------------------------------------------------------------------------------------------------
#                 A 1ª CATEGORIA E AS SEGUINTES: A RÉGUA PERTO DE COMO CONCORDAMOS ENTRE NÓS?
# ----------------------------------------------------------------------------------------------------
def first_and_following_hits(pairs_of_lists):
    first_hits, first_total, following_hits, following_total = 0, 0, 0, 0             # contadores
    for answer_list, reference_list in pairs_of_lists:
        if not answer_list:
            continue                                                                  # resposta sem categoria não tem 1ª nem seguintes
        first_total += 1                                                              # uma 1ª categoria a conferir
        first_hits += answer_list[0] in reference_list                                # a 1ª está na outra lista?
        following_total += len(answer_list) - 1                                       # as categorias seguintes a conferir
        following_hits += sum(label in reference_list for label in answer_list[1:])   # quantas delas estão na outra lista
    return {"first_category_%": round(100 * first_hits / first_total), "following_categories_%": round(100 * following_hits / following_total), "first_categories": first_total, "following_categories": following_total}

between_us = []                                                                           # cada resposta de um de nós contra a do colega, nas 50 atividades em dobro
for _, pair in pair_rows.iterrows():
    between_us.append((split_labels(pair["labels_a"]), split_labels(pair["labels_b"])))   # a do primeiro contra a do segundo
    between_us.append((split_labels(pair["labels_b"]), split_labels(pair["labels_a"])))   # e o contrário

ruler_against_gold = [(split_labels(ruler_labels[activity_id]), split_labels(gold_by_activity.loc[activity_id, "gold_labels"])) for activity_id in sampled_ids]   # a lista da régua contra o gabarito

different_amount = sum(len(split_labels(pair["labels_a"])) != len(split_labels(pair["labels_b"])) for _, pair in pair_rows.iterrows())                                                     # atividades em dobro em que os dois deram quantidades diferentes
labels_per_answer = human_labels.assign(number_of_labels=human_labels["labels"].apply(lambda labels: len(split_labels(labels)))).groupby("reviewer")["number_of_labels"].mean().round(2)   # média de categorias por resposta de cada um
ruler_with_four = (activities_multilabel["number_of_labels"] >= 4).sum()                                                                                                                   # atividades a que a régua deu 4 categorias, nas 693
ruler_with_four_here = sum(len(split_labels(ruler_labels[activity_id])) >= 4 for activity_id in sampled_ids)                                                                               # e nestas 100
answers_with_four = sum(len(split_labels(labels)) >= 4 for labels in human_labels["labels"])                                                                                               # nossas respostas com 4 categorias
merged_with_four = sum(len(set(split_labels(pair["labels_a"])) | set(split_labels(pair["labels_b"]))) >= 4 for _, pair in pair_rows.iterrows())                                            # atividades em dobro em que as duas listas juntas chegam a 4

print("Atividades em dobro em que demos quantidades diferentes de categorias:", different_amount, "de", len(pair_rows))
print("Média de categorias por resposta:", labels_per_answer.to_dict())
print("Com 4 categorias | régua:", ruler_with_four, "de", len(activities_multilabel), "atividades (", ruler_with_four_here, "nestas 100 ) | nossas respostas:", answers_with_four, "de", len(human_labels), "| nossas duas listas juntas:", merged_with_four, "de", len(pair_rows))
pd.DataFrame({"entre nós: a categoria de um está na lista do colega": first_and_following_hits(between_us), "régua: a categoria dela está no gabarito": first_and_following_hits(ruler_against_gold)}).T

Atividades em dobro em que demos quantidades diferentes de categorias: 30 de 50
Média de categorias por resposta: {'Beatriz': 2.17, 'Bruna': 1.67, 'Henrique': 1.63, 'João': 1.7, 'Júlia': 2.17}
Com 4 categorias | régua: 1 de 693 atividades ( 0 nestas 100 ) | nossas respostas: 9 de 150 | nossas duas listas juntas: 11 de 50


,first_category_%,following_categories_%,first_categories,following_categories
entre nós: a categoria de um está na lista do colega,86,53,99,93
régua: a categoria dela está no gabarito,82,52,100,65


**O que deu.** A 1ª categoria da régua está entre as 1ªs válidas do gabarito em 68 das 100 atividades, e
em alguma posição do gabarito em 82. Ficar só com ela (como na Parte 2) acerta muito do que diz, com 82%
de precisão, mas cobre só 42% das categorias que demos no gabarito, enquanto a lista inteira, da Parte
3, troca um pouco de precisão (70%) por bem mais cobertura (60%). O modelo de linguagem, que leu cada
descrição, ficou acima da régua nas duas coisas, o que indica que ainda existe espaço entre uma conta de
proximidade e uma leitura de verdade.

Separando a 1ª categoria das seguintes, a régua fica perto de nós mesmos. A 1ª categoria dela está no
gabarito em 82% das vezes, e entre nós a 1ª categoria de um está na lista do colega em 86%. Nas
categorias seguintes, as da régua estão no gabarito em 52% das vezes, e entre nós a categoria seguinte
que um deu aparece na lista do colega em 53%. Isso acontece porque variamos bastante na quantidade de
categorias: em 30 das 50 atividades em dobro, os dois deram quantidades diferentes, e a média vai de
1,63 a 2,17 categorias por resposta, dependendo de quem respondeu, então é comum um de nós acrescentar
uma categoria seguinte em que o outro não pensou. Além disso, a régua raramente dá 4 categorias (1 das
693 atividades, nenhuma destas 100), enquanto nós demos 4 em 9 das 150 respostas, e as nossas duas
listas juntas chegam a 4 em 11 das 50 atividades em dobro. Mesmo assim, a régua acerta as seguintes mais
ou menos na mesma proporção em que elas coincidem entre nós. A comparação favorece um pouco a régua,
porque nas atividades vistas em dobro o gabarito junta as listas de dois de nós.

As três atividades sem categoria no gabarito contam como erro da régua, que sempre dá pelo menos uma
categoria, e elas voltam no notebook 10, no aviso de revisão humana.